# Logistic regression — Version 5

This notebook refits the best Version 4 trial with its chosen inputs. The last section removes weakest inputs and refits once to see if we can find improvements there.


Otherwise, this notebook really doesn't do anything that new.

## A. Import data and dependencies

In [10]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from preprocessing import import_data, group_diagnoses, group_medications, engineer
from scoring import score

Xt_raw, yt, Xv_raw, yv = import_data()

Importing train/validation data...
Finished the import.


## B. Version 4 result and selected inputs

In [11]:
BEST_TRIAL = 3410
V4_F2 = 0.439513
THRESHOLD = 0.4483490425281894
C = 1.5036376551308464
best_features = """
race age admission_type_id payer_code medical_specialty
metformin repaglinide nateglinide chlorpropamide glimepiride
glipizide glyburide tolbutamide pioglitazone rosiglitazone
acarbose miglitol metformin-pioglitazone diabetesmed alpha_glucosidase_inhibitor
biguanide sulfonylurea_gen2 thiazolidinedione number_outpatient_log1p number_emergency_log1p
number_inpatient_log1p eng_prior_inpatient_any eng_inpatient_emergency_interaction eng_labs_per_day eng_acute_visits
eng_any_prior_visit eng_acute_visit_share number_emergency_bin number_emergency_2plus number_inpatient_2plus
number_inpatient_5plus time_in_hospital_log1p time_in_hospital_sqrt time_in_hospital_squared num_lab_procedures_log1p
num_lab_procedures_sqrt num_lab_procedures_squared num_procedures_log1p num_procedures_sqrt num_medications_log1p
num_medications_sqrt number_outpatient_sqrt number_emergency_squared number_inpatient_sqrt number_diagnoses_log1p
number_diagnoses_sqrt number_diagnoses_squared time_in_hospital_x_num_lab_procedures time_in_hospital_x_num_procedures time_in_hospital_x_number_outpatient
time_in_hospital_x_number_emergency time_in_hospital_x_number_inpatient num_lab_procedures_x_num_procedures num_lab_procedures_x_number_emergency num_lab_procedures_x_number_inpatient
num_procedures_x_num_medications num_procedures_x_number_inpatient num_procedures_x_number_diagnoses num_medications_x_number_outpatient num_medications_x_number_inpatient
num_medications_x_number_diagnoses number_outpatient_x_number_inpatient number_emergency_x_number_inpatient number_emergency_x_number_diagnoses number_inpatient_x_number_diagnoses
num_lab_procedures_per_day eng_labs_per_medication eng_drugs_No eng_drugs_Steady eng_drugs_Up
eng_drugs_Down eng_drugs_changed metformin_changed repaglinide_active nateglinide_changed
chlorpropamide_active glimepiride_active acetohexamide_active acetohexamide_changed glipizide_changed
tolbutamide_active tolbutamide_changed pioglitazone_active pioglitazone_changed miglitol_changed
troglitazone_active examide_changed citoglipton_active insulin_changed glyburide-metformin_changed
glipizide-metformin_changed glimepiride-pioglitazone_active glimepiride-pioglitazone_changed metformin-pioglitazone_active metformin-pioglitazone_changed
meglitinide_count_Steady meglitinide_count_Up meglitinide_count_active sulfonylurea_gen1_count_Steady sulfonylurea_gen2_count_Steady
sulfonylurea_gen2_count_Up sulfonylurea_gen2_count_Down thiazolidinedione_count_Up alpha_glucosidase_count_active metformin_combinations_count_Up
metformin_combinations_count_Down metformin_combinations_count_active eng_distinct_diagnosis_groups eng_diagnosis_count_Circulatory eng_age_squared
eng_age_50plus eng_age_85plus diag_1_group_with_insulin diag_1_group_with_discharge_disposition_id age_with_discharge_disposition_id
insulin_with_change admission_type_id_with_admission_source_id diag_1_group_with_diag_2_group diabetesmed_with_a1cresult
""".split()

display(pd.DataFrame({"trial": [BEST_TRIAL], "validation_F2": [V4_F2],
                      "threshold": [THRESHOLD], "C": [C],
                      "class_weight": ["balanced"], "inputs": [len(best_features)]}))
display(pd.DataFrame({"selected_input": best_features}))

,trial,validation_F2,threshold,C,class_weight,inputs
0,3410,0.439513,0.448349,1.503638,balanced,124


,selected_input
0,race
1,age
2,admission_type_id
3,payer_code
4,medical_specialty
...,...
119,age_with_discharge_disposition_id
120,insulin_with_change
121,admission_type_id_with_admission_source_id
122,diag_1_group_with_diag_2_group


## C. Engineer the inputs selected by Version 4

In [12]:
visits = ["number_outpatient", "number_emergency", "number_inpatient"]
base_numeric = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
                *visits, "number_diagnoses"]
classes = {
    "meglitinide": ["repaglinide", "nateglinide"],
    "sulfonylurea_gen1": ["chlorpropamide", "acetohexamide", "tolbutamide", "tolazamide"],
    "sulfonylurea_gen2": ["glimepiride", "glipizide", "glyburide"],
    "thiazolidinedione": ["pioglitazone", "rosiglitazone", "troglitazone"],
    "alpha_glucosidase": ["acarbose", "miglitol"],
    "metformin_combinations": ["glyburide-metformin", "glipizide-metformin",
                               "metformin-rosiglitazone", "metformin-pioglitazone"],
}
frames = []
for raw in (Xt_raw, Xv_raw):
    X = raw.drop(columns=["encounter_id", "readmitted", "target", "patient_nbr"], errors="ignore").copy()
    grouped_diag, grouped_med = group_diagnoses(X), group_medications(X)
    for col in ("diag_1", "diag_2", "diag_3"):
        X[col + "_group"] = grouped_diag[col]
    for col in ("alpha_glucosidase_inhibitor", "biguanide", "sulfonylurea_gen2", "thiazolidinedione"):
        X[col] = grouped_med[col]
    X = engineer(X)

    total = X[visits].sum(axis=1, min_count=3)
    acute = X["number_inpatient"] + X["number_emergency"]
    extra = {"eng_acute_visits": acute, "eng_any_prior_visit": total.gt(0).astype(int),
             "eng_acute_visit_share": acute / (1 + total),
             "number_emergency_bin": pd.cut(X["number_emergency"], [-1, 0, 1, 2, 4, np.inf],
                                            labels=["0", "1", "2", "3-4", "5+"]).astype(object),
             "number_emergency_2plus": X["number_emergency"].ge(2).astype(int)}
    for n in (2, 5):
        extra[f"number_inpatient_{n}plus"] = X["number_inpatient"].ge(n).astype(int)

    for col in base_numeric:
        z = X[col].clip(lower=0)
        extra[col + "_log1p"] = np.log1p(z)
        extra[col + "_sqrt"] = np.sqrt(z)
        extra[col + "_squared"] = z ** 2
    for i, a in enumerate(base_numeric):
        for b in base_numeric[i + 1:]:
            name = a + "_x_" + b
            if name in best_features:
                extra[name] = extra[a + "_log1p"] * extra[b + "_log1p"]
    extra["num_lab_procedures_per_day"] = X["num_lab_procedures"] / X["time_in_hospital"].clip(lower=1)
    extra["eng_labs_per_medication"] = X["num_lab_procedures"] / (1 + X["num_medications"])

    drugs = raw.loc[:, "metformin":"metformin-pioglitazone"].columns.tolist()
    for status in ("No", "Steady", "Up", "Down"):
        extra["eng_drugs_" + status] = X[drugs].eq(status).sum(axis=1)
    extra["eng_drugs_changed"] = X[drugs].isin(["Up", "Down"]).sum(axis=1)
    for name in best_features:
        if name.endswith(("_active", "_changed")) and name.rsplit("_", 1)[0] in drugs:
            drug, mode = name.rsplit("_", 1)
            extra[name] = X[drug].isin(["Steady", "Up", "Down"] if mode == "active" else ["Up", "Down"]).astype(int)
    for name, members in classes.items():
        for status in ("Steady", "Up", "Down"):
            key = name + "_count_" + status
            if key in best_features:
                extra[key] = X[members].eq(status).sum(axis=1)
        key = name + "_count_active"
        if key in best_features:
            extra[key] = X[members].isin(["Steady", "Up", "Down"]).sum(axis=1)

    diag_groups = X[["diag_1_group", "diag_2_group", "diag_3_group"]]
    extra["eng_distinct_diagnosis_groups"] = diag_groups.nunique(axis=1)
    extra["eng_diagnosis_count_Circulatory"] = diag_groups.eq("Circulatory").sum(axis=1)
    age_mid = pd.to_numeric(X["age"].astype("string").str.extract(r"\[(\d+)-", expand=False), errors="coerce") + 5
    extra["eng_age_squared"] = age_mid ** 2
    for cutoff in (50, 85):
        extra[f"eng_age_{cutoff}plus"] = age_mid.ge(cutoff).astype(int)
    for a, b in (("diag_1_group", "insulin"), ("diag_1_group", "discharge_disposition_id"),
                 ("age", "discharge_disposition_id"), ("insulin", "change"),
                 ("admission_type_id", "admission_source_id"), ("diag_1_group", "diag_2_group"),
                 ("diabetesmed", "a1cresult")):
        extra[a + "_with_" + b] = X[a].fillna("Missing").astype(str) + "|" + X[b].fillna("Missing").astype(str)
    X = pd.concat([X, pd.DataFrame({k: v for k, v in extra.items() if k not in X.columns}, index=X.index)], axis=1)
    for col in ("diag_1", "diag_2", "diag_3", "admission_type_id",
                "discharge_disposition_id", "admission_source_id"):
        X[col] = X[col].astype(str).where(X[col].notna(), np.nan)
    for col in X.select_dtypes(exclude="number"):
        X[col] = X[col].astype(str).where(X[col].notna(), np.nan)
    frames.append(X)
Xt, Xv = frames

## D. Preprocessing and model

In [13]:
numeric = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True,
                                             add_indicator=True)),
                    ("scale", StandardScaler())])
categorical = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Missing",
                                                 keep_empty_features=True)),
                        ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                 min_frequency=20))])

def make_pipeline(columns):
    prep = ColumnTransformer([(f"feature_{i}", numeric if pd.api.types.is_numeric_dtype(Xt[col])
                               else categorical, [col]) for i, col in enumerate(columns)],
                             sparse_threshold=1.0)
    model = LogisticRegression(solver="liblinear", C=C, class_weight="balanced",
                               max_iter=300, tol=1e-3)
    return Pipeline([("preprocess", prep), ("model", model)])

## E. Select only the 124 winning inputs

In [14]:
Xt_best, Xv_best = Xt[best_features], Xv[best_features]
print(f"{Xt_best.shape[1]} selected inputs, {len(Xt_best):,} training and {len(Xv_best):,} validation encounters")

124 selected inputs, 65,129 training and 16,283 validation encounters


## F. Fit the winning logistic regression

In [15]:
full = make_pipeline(best_features)
full.fit(Xt_best, yt)
full_prob = full.predict_proba(Xv_best)[:, 1]

## G. Score on validation

In [16]:
score(yv, proba=full_prob, threshold=THRESHOLD)

Accuracy: 0.565
Confusion matrix:
 [[7861 6605]
 [ 480 1337]]
              precision    recall  f1-score   support

           0       0.94      0.54      0.69     14466
           1       0.17      0.74      0.27      1817

    accuracy                           0.56     16283
   macro avg       0.56      0.64      0.48     16283
weighted avg       0.86      0.56      0.64     16283

F1 @0.4483490425281894: 0.274
F2 @0.4483490425281894: 0.44
ROC-AUC: 0.6821
PR-AUC:  0.2186

Best-threshold F1: 0.28865 at threshold 0.54671.

Best-threshold F2: 0.43951 at threshold 0.44835.


## H. Encoded weights and input importance

A positive coefficient raises the log odds of 30-day readmission. The table shows the 30 largest absolute weights of the *encoded* columns after imputation and one-hot encoding. The complete table remains in `weights`. 

In [17]:
prep = full.named_steps["preprocess"]
coef = full.named_steps["model"].coef_[0]
weights = pd.DataFrame({"encoded_feature": prep.get_feature_names_out(), "coefficient": coef})
weights["absolute_weight"] = weights["coefficient"].abs()
display(weights.sort_values("absolute_weight", ascending=False).head(30))

transformed_train = prep.transform(Xt_best)
margin = full.decision_function(Xt_best)
cutoff = np.log(THRESHOLD / (1 - THRESHOLD))
train_f2 = fbeta_score(yt, margin >= cutoff, beta=2)
importance = pd.Series({
    col: train_f2 - fbeta_score(
        yt, margin - transformed_train[:, prep.output_indices_[f"feature_{i}"]]
        @ coef[prep.output_indices_[f"feature_{i}"]] >= cutoff, beta=2)
    for i, col in enumerate(best_features)
}, name="training_F2_drop").sort_values(ascending=False)
display(importance.to_frame().head(30))

,encoded_feature,coefficient,absolute_weight
443,feature_119__age_with_discharge_disposition_id...,-2.828102,2.828102
457,feature_119__age_with_discharge_disposition_id...,-2.705072,2.705072
378,feature_118__diag_1_group_with_discharge_dispo...,-2.568195,2.568195
286,feature_118__diag_1_group_with_discharge_dispo...,-2.356440,2.356440
376,feature_118__diag_1_group_with_discharge_dispo...,-2.315116,2.315116
429,feature_119__age_with_discharge_disposition_id...,-2.291865,2.291865
468,feature_119__age_with_discharge_disposition_id...,-2.075725,2.075725
362,feature_118__diag_1_group_with_discharge_dispo...,-2.025665,2.025665
45,feature_4__medical_specialty_Gynecology,-1.979286,1.979286
361,feature_118__diag_1_group_with_discharge_dispo...,1.704224,1.704224


,training_F2_drop
number_inpatient_sqrt,0.151295
number_emergency_log1p,0.035455
num_procedures_log1p,0.029298
diag_1_group_with_discharge_disposition_id,0.028594
time_in_hospital_sqrt,0.028407
num_procedures_x_num_medications,0.021872
number_emergency_bin,0.018917
admission_type_id_with_admission_source_id,0.015660
num_procedures_sqrt,0.014080
miglitol,0.013893


## I. Drop the ten least helpful inputs and refit

In [18]:
removed = importance.tail(20).index.tolist()
remaining = [col for col in best_features if col not in removed]
reduced = make_pipeline(remaining)
reduced.fit(Xt[remaining], yt)
print("Removed:", removed)
score(yv, proba=reduced.predict_proba(Xv[remaining])[:, 1], threshold=THRESHOLD)

Removed: ['eng_any_prior_visit', 'metformin_combinations_count_active', 'glipizide_changed', 'pioglitazone_changed', 'insulin_with_change', 'number_inpatient_5plus', 'num_procedures_x_number_inpatient', 'metformin_changed', 'eng_distinct_diagnosis_groups', 'glimepiride', 'metformin', 'biguanide', 'tolbutamide', 'alpha_glucosidase_inhibitor', 'chlorpropamide', 'repaglinide', 'metformin-pioglitazone', 'nateglinide', 'pioglitazone', 'glipizide']
Accuracy: 0.565
Confusion matrix:
 [[7871 6595]
 [ 496 1321]]
              precision    recall  f1-score   support

           0       0.94      0.54      0.69     14466
           1       0.17      0.73      0.27      1817

    accuracy                           0.56     16283
   macro avg       0.55      0.64      0.48     16283
weighted avg       0.85      0.56      0.64     16283

F1 @0.4483490425281894: 0.271
F2 @0.4483490425281894: 0.435
ROC-AUC: 0.6821
PR-AUC:  0.2186

Best-threshold F1: 0.28808 at threshold 0.54892.

Best-threshold F2: 0.